In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Implement a GPU program to calculate the categorical cross-entropy loss for a batch of predictions.
  Given a matrix of predicted logits $Z$ of size $N \times C$ and a vector of true class labels <code>true_labels</code> of size $N$, compute the average cross-entropy loss over the batch.
  The loss for a single sample $j$ with logits $z_j = [z_{j1}, \ldots, z_{jC}]$ and true label $y_j$ is calculated using the numerically stable formula:
  $$ \text{Loss}_j = \log\left(\sum_{k=1}^{C} e^{z_{jk}}\right) - z_{j, y_j} $$
  The final output stored in the <code>loss</code> variable should be the average loss over the $N$ samples:
  $$ L = \frac{1}{N} \sum_{j=1}^{N} \text{Loss}_j $$
  The input parameters are <code>logits</code>, <code>true_labels</code>, <code>N</code> (number of samples), and <code>C</code> (number of classes). The result should be stored in <code>loss</code> (a pointer to a single float).
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>External libraries are not permitted</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final result (average loss) must be stored in <code>loss</code></li>
</ul>

<h2>Example 1:</h2>
<pre>Input:  N = 2, C = 3
        logits = [[1.0, 2.0, 0.5], [0.1, 3.0, 1.5]]
        true_labels = [1, 1]
Output: loss = [0.3548926]</pre>


<h2>Example 2:</h2>
<pre>Input:  N = 3, C = 4
        logits = [[-0.5, 1.5, 0.0, 1.0], [2.0, -1.0, 0.5, 0.5], [0.0, 0.0, 0.0, 0.0]]
        true_labels = [3, 0, 1]
Output: loss = [0.98820376]</pre>

<h2>Constraints</h2>
<ul>
  <li>1 &le; <code>N</code> &le; 10,000</li>
  <li>2 &le; <code>C</code> &le; 1,000</li>
  <li>-10.0 &le; <code>logits[i, j]</code> &le; 10.0</li>
  <li>0 &le; <code>true_labels[i]</code> &le; <code>C</code></li>

  <li>Performance is measured with <code>N</code> = 10,000</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// logits, true_labels, loss are device pointers
extern "C" void solve(const float* logits, const int* true_labels, float* loss, int N, int C) {}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# logits, true_labels, loss are tensors on the GPU
@cute.jit
def solve(
    logits: cute.Tensor, true_labels: cute.Tensor, loss: cute.Tensor, N: cute.Int32, C: cute.Int32
):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# logits, true_labels are tensors on the GPU
@jax.jit
def solve(logits: jax.Array, true_labels: jax.Array, N: int, C: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


@export
def solve(
    logits: UnsafePointer[Float32, MutExternalOrigin],
    true_labels: UnsafePointer[Int32, MutExternalOrigin],
    loss: UnsafePointer[Float32, MutExternalOrigin],
    N: Int32,
    C: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


# logits, true_labels, loss are tensors on the GPU
def solve(logits: torch.Tensor, true_labels: torch.Tensor, loss: torch.Tensor, N: int, C: int):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


# logits, true_labels, loss are tensors on the GPU
def solve(logits: torch.Tensor, true_labels: torch.Tensor, loss: torch.Tensor, N: int, C: int):
    pass


# Evaluate Setup

In [ ]:
# --- Core Challenge Base ---
from abc import ABC, abstractmethod
from typing import Any, Dict, List


class ChallengeBase(ABC):
    def __init__(self, name: str, atol: float, rtol: float, num_gpus: int, access_tier: str):
        self.name = name
        self.atol = atol
        self.rtol = rtol
        self.num_gpus = num_gpus
        self.access_tier = access_tier

    @abstractmethod
    def reference_impl(self, *args, **kwargs):
        """
        Reference solution implementation.
        """
        pass

    @abstractmethod
    def get_solve_signature(self) -> Dict[str, Any]:
        """
        Get the function signature for solution.

        Returns:
            Dictionary with argtypes and restype for ctypes
        """
        pass

    @abstractmethod
    def generate_example_test(self) -> List[Dict[str, Any]]:
        """
        Generate an example test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass

    @abstractmethod
    def generate_functional_test(self) -> List[Dict[str, Any]]:
        """
        Generate functional test cases for this problem.

        Returns:
            List of test case dictionaries
        """
        pass

    @abstractmethod
    def generate_performance_test(self) -> List[Dict[str, Any]]:
        """
        Generate a performance test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass


# --- Challenge Logic ---
import ctypes
from typing import Any, Dict, List

import torch


class Challenge(ChallengeBase):
    def __init__(self):
        super().__init__(
            name="Categorical Cross Entropy Loss",
            atol=1e-05,
            rtol=1e-05,
            num_gpus=1,
            access_tier="free",
        )

    def reference_impl(
        self, logits: torch.Tensor, true_labels: torch.Tensor, loss: torch.Tensor, N: int, C: int
    ):
        assert logits.dtype == torch.float32
        assert true_labels.dtype == torch.int32
        assert loss.dtype == torch.float32
        assert logits.shape == (N, C)
        assert true_labels.shape == (N,)
        assert loss.shape == (1,)
        assert N > 0 and C > 0
        total_loss = 0.0
        for i in range(N):
            log_probs = logits[i]
            true_label = true_labels[i].item()
            assert 0 <= true_label < C
            max_logit = torch.max(log_probs)
            log_sum_exp = max_logit + torch.log(torch.sum(torch.exp(log_probs - max_logit)))
            loss_i = log_sum_exp - log_probs[true_label]
            total_loss += loss_i.item()
        loss[0] = total_loss / N

    def get_solve_signature(self) -> Dict[str, tuple]:
        return {
            "logits": (ctypes.POINTER(ctypes.c_float), "in"),
            "true_labels": (ctypes.POINTER(ctypes.c_int), "in"),
            "loss": (ctypes.POINTER(ctypes.c_float), "out"),
            "N": (ctypes.c_int, "in"),
            "C": (ctypes.c_int, "in"),
        }

    def generate_example_test(self) -> Dict[str, Any]:
        dtype_logits = torch.float32
        dtype_labels = torch.int32
        logits = torch.tensor([[1.0, 2.0, 0.5], [0.1, 3.0, 1.5]], device="cuda", dtype=dtype_logits)
        true_labels = torch.tensor([1, 1], device="cuda", dtype=dtype_labels)
        loss = torch.zeros(1, device="cuda", dtype=dtype_logits)
        return {
            "logits": logits,
            "true_labels": true_labels,
            "loss": loss,
            "N": 2,
            "C": 3,
        }

    def generate_functional_test(self) -> List[Dict[str, Any]]:
        dtype_logits = torch.float32
        dtype_labels = torch.int32
        tests = []
        # basic_example
        tests.append(
            {
                "logits": torch.tensor(
                    [[1.0, 2.0, 0.5], [0.1, 3.0, 1.5]], device="cuda", dtype=dtype_logits
                ),
                "true_labels": torch.tensor([1, 1], device="cuda", dtype=dtype_labels),
                "loss": torch.zeros(1, device="cuda", dtype=dtype_logits),
                "N": 2,
                "C": 3,
            }
        )
        # example_2
        tests.append(
            {
                "logits": torch.tensor(
                    [[-0.5, 1.5, 0.0, 1.0], [2.0, -1.0, 0.5, 0.5], [0.0, 0.0, 0.0, 0.0]],
                    device="cuda",
                    dtype=dtype_logits,
                ),
                "true_labels": torch.tensor([3, 0, 1], device="cuda", dtype=dtype_labels),
                "loss": torch.zeros(1, device="cuda", dtype=dtype_logits),
                "N": 3,
                "C": 4,
            }
        )
        # single_sample
        tests.append(
            {
                "logits": torch.tensor(
                    [[0.1, 0.2, 0.3, 0.4, 0.5]], device="cuda", dtype=dtype_logits
                ),
                "true_labels": torch.tensor([4], device="cuda", dtype=dtype_labels),
                "loss": torch.zeros(1, device="cuda", dtype=dtype_logits),
                "N": 1,
                "C": 5,
            }
        )
        # uniform_logits_correct_label
        tests.append(
            {
                "logits": torch.tensor([[1.0] * 5, [1.0] * 5], device="cuda", dtype=dtype_logits),
                "true_labels": torch.tensor([0, 0], device="cuda", dtype=dtype_labels),
                "loss": torch.zeros(1, device="cuda", dtype=dtype_logits),
                "N": 2,
                "C": 5,
            }
        )
        # high_confidence_correct
        tests.append(
            {
                "logits": torch.tensor(
                    [[-5.0, -5.0, 10.0, -5.0], [10.0, -5.0, -5.0, -5.0]],
                    device="cuda",
                    dtype=dtype_logits,
                ),
                "true_labels": torch.tensor([2, 0], device="cuda", dtype=dtype_labels),
                "loss": torch.zeros(1, device="cuda", dtype=dtype_logits),
                "N": 2,
                "C": 4,
            }
        )
        # high_confidence_incorrect
        tests.append(
            {
                "logits": torch.tensor(
                    [[10.0, -5.0, -5.0], [-5.0, 10.0, -5.0]], device="cuda", dtype=dtype_logits
                ),
                "true_labels": torch.tensor([1, 2], device="cuda", dtype=dtype_labels),
                "loss": torch.zeros(1, device="cuda", dtype=dtype_logits),
                "N": 2,
                "C": 3,
            }
        )
        # larger_batch_random
        tests.append(
            {
                "logits": torch.empty(100, 5, device="cuda", dtype=dtype_logits).uniform_(
                    -5.0, 5.0
                ),
                "true_labels": torch.randint(0, 5, (100,), device="cuda", dtype=dtype_labels),
                "loss": torch.zeros(1, device="cuda", dtype=dtype_logits),
                "N": 100,
                "C": 5,
            }
        )
        return tests

    def generate_performance_test(self) -> Dict[str, Any]:
        dtype_logits = torch.float32
        dtype_labels = torch.int32
        logits = torch.empty(10000, 1000, device="cuda", dtype=dtype_logits).uniform_(-10.0, 10.0)
        true_labels = torch.randint(0, 1000, (10000,), device="cuda", dtype=dtype_labels)
        loss = torch.zeros(1, device="cuda", dtype=dtype_logits)
        return {
            "logits": logits,
            "true_labels": true_labels,
            "loss": loss,
            "N": 10000,
            "C": 1000,
        }


ch = Challenge()



In [ ]:
import os
import time
import ctypes
import torch

class Evaluate:
    @staticmethod
    def eval_cuda(ch):
        # 1. Compile a fresh uniquely named library
        so_filename = f'solution_func_{int(time.time())}.so'
        os.system(f'nvcc -shared -Xcompiler -fPIC -O3 solution.cu -o {so_filename}')
        lib = ctypes.CDLL(f'./{so_filename}')
        
        # 2. Extract signature and set argtypes
        signature = ch.get_solve_signature()
        lib.solve.argtypes = [arg_info[0] for arg_info in signature.values()]
        
        Evaluate._run_tests(ch, signature, lambda kwargs: lib.solve(*Evaluate._build_cuda_args(kwargs, signature)))

    @staticmethod
    def eval_python(ch):
        import importlib.util
        import sys
        
        spec = importlib.util.spec_from_file_location("solution", "solution.py")
        solution = importlib.util.module_from_spec(spec)
        sys.modules["solution"] = solution
        spec.loader.exec_module(solution)
        
        signature = ch.get_solve_signature()
        Evaluate._run_tests(ch, signature, lambda kwargs: Evaluate._run_python(solution, kwargs))

    @staticmethod
    def _run_python(solution, kwargs):
        solution.solve(**kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    @staticmethod
    def eval_mojo(ch):
        print("Mojo evaluation is currently executed via a separate runner or wrapper.")
        print("Ensure you have the mojo compiler installed and use 'mojo build solution.mojo' + ctypes/ffi,")
        print("or run an external python bridge. This is a stub.")

    @staticmethod
    def _build_cuda_args(kwargs, signature):
        cuda_args = []
        for k, (arg_type, dir_type) in signature.items():
            val = kwargs[k]
            if isinstance(val, torch.Tensor):
                cuda_args.append(ctypes.cast(val.data_ptr(), arg_type))
            else:
                cuda_args.append(arg_type(val))
        return cuda_args

    @staticmethod
    def _run_tests(ch, signature, run_fn):
        print("=== Running Functional Tests ===")
        functional_tests = ch.generate_functional_test()
        all_passed = True
        
        for i, test in enumerate(functional_tests):
            ref_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            test_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            
            # Run Reference
            ch.reference_impl(**ref_kwargs)
            
            # Run implementation
            run_fn(test_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Verify outputs
            match = True
            for k, (_, dir_type) in signature.items():
                if dir_type == "out":
                    if not torch.allclose(ref_kwargs[k], test_kwargs[k], atol=ch.atol, rtol=ch.rtol):
                        match = False
                        print(f"❌ Test {i+1}/{len(functional_tests)} Failed on output '{k}'")
                        break
            
            if match:
                print(f"✅ Test {i+1}/{len(functional_tests)} Passed")
            else:
                all_passed = False
                break
                
        if all_passed:
            print("\n🎉 All functional tests passed!")
            return True
        else:
            return False



# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
